## Importing libraries and data

In [ ]:
import numpy as np
import pandas as pd
import pickle

In [ ]:
cols_df1 = ['pseudopaz', 'patient_sex']
df1 = pd.read_excel('../raw_data-originals/dati_escribe.xlsx',
    usecols=cols_df1,
    dtype_backend='numpy_nullable',
    dtype={col: 'string' for col in cols_df1}
)

cols_df2 = ['PSEUDO_PAZ', 'SESSO_COD']
df2 = pd.read_excel('../raw_data-originals/Estrazioni_iconsulting.xlsx', sheet_name='Ricoveri',
    usecols=cols_df2,
    dtype_backend='numpy_nullable',
    dtype={col: 'string' for col in cols_df2}
)

## Simplifying dfs

drop df2 values where pseudopaz is not in df1

In [ ]:
ppaz_1 = df1['pseudopaz'].drop_duplicates().tolist()

In [ ]:
df2 = df2[df2['PSEUDO_PAZ'].isin(ppaz_1)]

Drop the duplicated rows and reset index

In [ ]:
df1_simple = df1[['pseudopaz', 'patient_sex']].drop_duplicates()
df2_simple = df2[['PSEUDO_PAZ', 'SESSO_COD']].drop_duplicates()

In [ ]:
df1_simple = df1_simple.sort_values(by=['pseudopaz'])
df2_simple = df2_simple.sort_values(by=['PSEUDO_PAZ'])

In [ ]:
df1_simple = df1_simple.reset_index(drop=True)
df2_simple = df2_simple.reset_index(drop=True)

# Analysis

## Consistency of codes between the two datasets

### First representation

In [ ]:
def counts_dict(series):
    return series.value_counts().to_dict()

def fmt_dict(d):
    return ', '.join(f'{k}: {v}' for k, v in d.items())

In [ ]:
escribe_data = {}
for pid, group in df1.groupby('pseudopaz'):
    escribe_data[pid] = counts_dict(group['patient_sex'])

iconsulting_data = {}
for pid, group in df2.groupby('PSEUDO_PAZ'):
    iconsulting_data[pid] = counts_dict(group['SESSO_COD'])

escribe_series     = pd.Series(escribe_data,     name='sex_escribe')
iconsulting_series = pd.Series(iconsulting_data, name='sex_iconsulting')

escribe_series.index.name     = 'pseudo-patient'
iconsulting_series.index.name = 'pseudo-patient'

summary = pd.concat([escribe_series, iconsulting_series], axis=1).reset_index()

summary['sex_escribe']     = summary['sex_escribe'].apply(fmt_dict)
summary['sex_iconsulting'] = summary['sex_iconsulting'].apply(fmt_dict)

In [ ]:
summary

### Second representation

In [ ]:
# Merge on patient ID
merged = df1.merge(df2, left_on="pseudopaz", right_on="PSEUDO_PAZ", how="left")
merged_simple = df1_simple.merge(df2_simple, left_on="pseudopaz", right_on="PSEUDO_PAZ", how="left")

In [ ]:
# Cross-tabulate: rows = df1 codes, columns = df2 codes
cross = pd.crosstab(merged["patient_sex"], merged["SESSO_COD"])
cross_simple = pd.crosstab(merged_simple["patient_sex"], merged_simple["SESSO_COD"])

cross.index.name = 'escribe' ; cross.columns.name = 'iconsulting'
cross_simple.index.name = 'escribe' ; cross_simple.columns.name = 'iconsulting'

cross_simple
# print('cross\n', cross)
# print('cross_simple\n', cross_simple)

### Excluding `2`s

Regarding df1 and excluding rows where sex is not `2`, which pseudopazs has more than one sex code

In [ ]:
df1_no2 = df1[df1["patient_sex"] != '2']

In [ ]:
incons = df1_no2.groupby('pseudopaz')['patient_sex'].nunique()
incons = incons[incons > 1].sort_values(ascending=False)
incons

In [ ]:
df1[df1['pseudopaz'] == '33794016081826308645769']
# df1_simple[df1_simple['pseudopaz'] == '33794016081826308645769']

## Export

In [ ]:
# # Parquet
# df.to_parquet('../raw_data/parquets/1-escribe.parquet')

# # Pickle
# df.to_pickle('../raw_data/pickles/1-escribe.pickle')
# df['pseudopaz'].to_pickle('../raw_data/pickles/pps/1-escribe-pp.pickle')
# df['pseudopaz'].drop_duplicates().to_pickle('../raw_data/pickles/pps/1-escribe-pp-uniq.pickle')
# with open('../raw_data/pickles/pps/1-escribe-pp-uniq-list.pickle', 'wb') as file:
#     pickle.dump(df['pseudopaz'].drop_duplicates().tolist(), file, protocol=pickle.HIGHEST_PROTOCOL)

# # Excel
# summary.to_excel('../raw_data/excel_formatted/escribe-sex_incons.xlsx', sheet_name='escribe-sex_incons', index=False)